# 09 · Doubly Linked List

A **doubly linked list (DLL)** node has **two** pointers: `next` (forward) and `prev` (backward).

```
None ← [1] ⇄ [2] ⇄ [3] → None
```

| | Singly | Doubly |
|---|---|---|
| Move backward | ❌ | ✅ |
| Delete a node knowing only that node | needs the previous node | uses `node.prev` |
| Memory per node | 1 pointer | 2 pointers |

**Golden rule:** whenever you change a `next` link, update the matching `prev` link too.

The code is exactly my original; I added explanations/comments and ⚠️ notes where a function has a problem.

**Topics:** DLL class · reverse · remove all occurrences of a key · pair with a given sum · remove duplicates from a sorted DLL

## 1. The `DoublyLinkedList` class
| Method | What it does |
|---|---|
| `Insert_at_head` | new node becomes head; the old head's `prev` points to it |
| `append` | walk to the tail, link both directions |
| `insert_at` | stop at the node before the position, then fix **four** pointers |
| `traverse` | print `a <-> b <-> c` |
| `delete_head` / `delete_last` | unlink at either end |
| `delete_in_between` | `node.prev.next = node.next` and `node.next.prev = node.prev` |

In [ ]:
class Node:
  def __init__(self, val):
    self.val = val
    self.next = None
    # backward pointer
    self.prev = None

class DoublyLinkedList:
  def __init__(self):
    self.head = None
    # (stores the class on the instance; not needed for the logic)
    self.Node = Node

  def Insert_at_head(self, val):
    new_node = Node(val)
    if not self.head:
      self.head = new_node
    else:
      # insert at head: point to the old head
      new_node.next = self.head
      # old head points back to the new node
      self.head.prev = new_node
      self.head = new_node

  def append(self, val):
    new_node = Node(val)
    if not self.head:
      self.head = new_node
    else:
      current = self.head
      # walk to the tail
      while current.next:
        current = current.next
      current.next = new_node
      # link back from the new node
      new_node.prev = current

  def insert_at(self, val, position):
    if position == 0:
      self.Insert_at_head(val)
      return
    new_node = Node(val)
    current = self.head
    count = 0
    # stop at the node BEFORE the position
    while current and count < position - 1:
      current = current.next
      count += 1
    if current is None:
      print("Position out of bounds")
      return
    # new node → following node
    new_node.next = current.next
    new_node.prev = current
    if current.next:
      # following node → new node (only if it exists)
      current.next.prev = new_node
    current.next = new_node

  def traverse(self):
    current = self.head
    elements = []
    while current:
      elements.append(str(current.val))
      current = current.next
    # print as 1 <-> 2 <-> 3
    print(" <-> ".join(elements) if elements else "Empty List")

  def delete_head(self):
    if not self.head:
      print("List is empty")
      return
    if not self.head.next:
      self.head = None
    else:
      self.head = self.head.next
      # the new head has nothing before it
      self.head.prev = None

  def delete_last(self):
    if not self.head:
      print("List is empty")
      return
    if not self.head.next:
      self.head = None
      return
    current = self.head
    while current.next:
      current = current.next
    # the DLL advantage: step back through prev and cut off the last node
    current.prev.next = None

  def delete_in_between(self, position):
    if not self.head:
      print("List is empty")
      return
    if position == 0:
      self.delete_head()
      return
    current = self.head
    count = 0
    while current and count < position:
      current = current.next
      count += 1
    if current is None:
      print("Position out of bounds")
      return
    if current.next is None:
      self.delete_last()
    else:
      # bypass the node going forward...
      current.prev.next = current.next
      # ...and going backward
      current.next.prev = current.prev

In [ ]:
dll=DoublyLinkedList()
dll.append(223)

### Testing every method
Each section prints the list after the operation, so you can follow what changes.

In [ ]:
# Create list and test all methods
dll = DoublyLinkedList()

print("--- 1. Append (223, 10, 20) ---")
# list: 223 <-> 10 <-> 20 (after the next two appends)
dll.append(223)
dll.append(10)
dll.append(20)
dll.traverse()

print("\n--- 2. Insert at head (5) ---")
# 5 goes to the front
dll.Insert_at_head(5)
dll.traverse()

print("\n--- 3. Insert at position 2 (15) ---")
# 15 inserted at index 2
dll.insert_at(15, 2)
dll.traverse()

print("\n--- 4. Delete Head ---")
# removes 5
dll.delete_head()
dll.traverse()

print("\n--- 5. Delete Last ---")
# removes 20
dll.delete_last()
dll.traverse()

print("\n--- 6. Delete In Between (position 1) ---")
# removes the node at index 1 (15)
dll.delete_in_between(1)
dll.traverse()

--- 1. Append (223, 10, 20) ---
223 <-> 10 <-> 20

--- 2. Insert at head (5) ---
5 <-> 223 <-> 10 <-> 20

--- 3. Insert at position 2 (15) ---
5 <-> 223 <-> 15 <-> 10 <-> 20

--- 4. Delete Head ---
223 <-> 15 <-> 10 <-> 20

--- 5. Delete Last ---
223 <-> 15 <-> 10

--- 6. Delete In Between (position 1) ---
223 <-> 10


## 2. Reverse a doubly linked list
**Approach 1 – stack (swap values):** push all values on a stack, then walk again overwriting each node's value with `pop()`. O(n) time, O(n) space.

In [ ]:
def reverse_double_linked_list(self,head,stack):
  temp=head
  while temp is not None:
    # first walk: collect the values
    stack.append(temp.val)
    temp=temp.next
  temp=head
  while temp is not None:
    # second walk: take them back in reverse order
    e=stack.pop()
    # overwrite the node values (the nodes themselves don't move)
    temp.val=e
    temp=temp.next
  return head

**Approach 2 – swap pointers (optimal):** for every node swap its `next` and `prev`. The last node processed (the old tail) becomes the new head. O(n) time, O(1) space.

> ⚠️ **Note:** the function ignores its `head` argument and uses `self.head`, and it raises an error for an empty list (`curr.next` on `None`).

In [ ]:
def reverse_DLL(self,head):
  curr=self.head
  # single node: already reversed
  if curr.next is None:
    return curr
  prev=None
  while curr is not None:
    # save the next node
    front=curr.next
    # next now points backward
    curr.next=prev
    # prev now points forward
    curr.prev=front
    prev=curr
    curr=front
  # the old tail is the new head
  return prev

## 3. Remove all occurrences of a key
Walk the list; whenever `temp.val == key`, unlink the node:
- the previous node's `next` skips it
- the next node's `prev` skips it
- if it was the **head**, move the head forward

> ⚠️ **Note:** after a deletion `prev = temp` still points at the **deleted** node. Using `temp.prev` (every node already knows its previous node) avoids needing a separate `prev` variable.

In [ ]:
def remove_all_occurence_key(self,head, key):
  # single-node list that matches → empty list
  if head.next is None and head.val==key:
    return None
  temp=head
  prev=None
  new_head=head
  while temp is not None:
    # node to delete
    if temp.val==key:
      if prev is not None:
        # previous node skips it
        prev.next=temp.next
      if temp.next is not None:
        # next node points back to the previous one
        temp.next.prev=prev
      if temp==new_head:
        # deleting the head moves the head forward
        new_head=new_head.next
    prev=temp
    temp=temp.next
  return new_head

## 4. Find pairs with a given sum
| Approach | Works on | Time | Space |
|---|---|---|---|
| Brute force (two nested walks) | any list | O(n²) | O(1) |
| Hash set (`sum − value`) | any list | O(n) | O(n) |
| **Two pointers** (`left` at head, `right` at tail) | **sorted** list | O(n) | **O(1)** |

In [ ]:
def find_pair_with_given_sum(self,head,sum):
  temp1=head
  list=[]
  while temp1 is not None:
    # pair temp1 with every later node
    temp2=temp1.next
    while temp2 is not None:
      # pair found
      if temp1.val+temp2.val==sum:
        list.append([temp1.val,temp2.val])
      temp2=temp2.next
    temp1=temp1.next

  return list


**Hash set:** for each value look up `sum − value` among the values already seen.

> ⚠️ **Note:** this function builds `list` but never returns it (no `return list` at the end), so calling it gives `None`.

In [ ]:
def find_pairs_with_given_sum(self,head,sum):
  temp=head
  my_set=set()
  list=[]
  while temp is not None:
    # the partner needed
    remaining=sum-temp.val
    # the partner was seen earlier
    if remaining in my_set:
      list.append([remaining,temp.val])

    # remember this value
    my_set.add(temp.val)
    temp=temp.next


**Two pointers on a sorted DLL:** `left` starts at the head, `right` at the tail. Sum too small → `left = left.next`; sum too big → `right = right.prev`; match → record and move both. Only a **doubly** linked list can move `right` backward.

> ⚠️ **Note:** the loop condition `left.val < right.val` skips pairs of equal values such as `(2, 2)`; comparing node positions (`left is not right`) would include them.

In [ ]:
def finding_pairs_with_given_sum(self,head,sum):
  result=[]
  left=head
  right=head
  # move right to the tail
  while right.next is not None:
    right=right.next
  # stop when the pointers cross
  while left is not None and right is not None and left.val < right.val:
    total=left.val+right.val
    if total==sum:
      result.append([left.val,right.val])
      # sum too small → move the left pointer forward
      left=left.next
      # sum too big → move the right pointer back
      right=right.prev
    elif total>sum:
      right=right.prev
    else:
      left=left.next
  return result


## 5. Remove duplicates from a sorted DLL
In a sorted list duplicates are neighbours: when `curr.val == curr.next.val`, unlink one of them and fix both `next` and `prev`.

> ⚠️ **Note:** this function is unfinished. `curr` is never advanced inside the loop, so the loop would not terminate, and the pointer updates use `curr.prev` where the node after `curr` should be unlinked.

In [ ]:
def remove_duplicates_from_sorted_Double_linked_list(self,head):
    curr=head
    while curr is not None and curr.next is not None:
      # neighbouring duplicates
      if curr.val==curr.next.val:
        if curr.prev == head:
          curr.prev=None
          head=curr
        curr.prev.prev.next=curr
        curr.prev=curr.prev.prev
    return head

---
### ✅ Key takeaways
- In a DLL **every link change touches two pointers** (`next` and `prev`).
- `prev` lets you delete a node without tracking the previous node, and lets you run **two pointers from both ends**.
- Save `temp.next` before unlinking `temp`.